<a href="https://colab.research.google.com/github/gankur-oss/TestRerank/blob/main/Rerank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall -y FlagEmbedding transformers sentence-transformers -q

!pip install -q \
FlagEmbedding==1.3.5 \
transformers==4.46.1 \
sentence-transformers==3.2.1 \
accelerate sentencepiece

In [2]:
# Install dependencies
!pip install pandas numpy fastapi uvicorn

In [3]:
import torch
from FlagEmbedding import FlagReranker
import time
from concurrent.futures import ThreadPoolExecutor

In [10]:
# CPU thread optimization

import os
import time

os.environ["OMP_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"

torch.set_num_threads(8)
torch.set_num_interop_threads(8)

In [11]:
# Ensure GPU is used
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Load optimized reranker (FP16 for T4)
reranker = FlagReranker(
    "BAAI/bge-reranker-base",
    use_fp16=True
)

Device: cuda


In [12]:
# 100 UNIQUE HOME IMPROVEMENT PRODUCT DESCRIPTIONS
documents = [
"Bosch 20V cordless drill with lithium battery designed for DIY home repair and drilling into wood and drywall",
"Makita brushless cordless impact driver ideal for driving long screws in carpentry and furniture assembly",
"DeWalt heavy duty hammer drill engineered for drilling into concrete brick and masonry surfaces",
"Black and Decker compact cordless drill kit with charger and battery for everyday household tasks",
"High pressure expandable garden hose pipe 50 feet flexible design for watering lawns and plants",
"Ceramic waterproof bathroom wall tiles 30x30 cm suitable for bathroom renovation and remodeling",
"Manual tile cutter tool designed for accurate cutting of porcelain and ceramic tiles",
"Stainless steel rainfall shower head with adjustable spray modes for modern bathroom upgrade",
"LED ceiling panel light energy efficient square lighting fixture for indoor home lighting",
"Remote controlled indoor ceiling fan with silent motor for bedroom and living room cooling",
"Electric circular saw with carbide blade for cutting wooden boards and plywood sheets",
"Paint roller kit with tray extension pole and microfiber roller for smooth wall painting",
"Adjustable wrench spanner tool set for plumbing pipe fittings and mechanical repairs",
"Cordless electric screwdriver set with interchangeable bits for furniture assembly",
"Wall mounted bathroom mirror cabinet with internal storage shelves",
"Outdoor waterproof LED wall light fixture for garden pathways and patios",
"Garden pruning shears with stainless steel blades for trimming plants and shrubs",
"Steel claw hammer with anti slip grip for carpentry and home construction",
"Electric tile cutter machine with water cooling system for professional tile installation",
"Compact electric air blower tool for cleaning dust from workshops and electronics",
"Cordless drill driver combo kit including drill batteries and charger",
"Portable shop vacuum cleaner designed for garage dust and debris removal",
"Heavy duty pipe wrench tool for plumbing repairs and tightening metal pipes",
"Garden sprinkler rotating water sprinkler system for lawn irrigation",
"Rechargeable LED work light portable lamp for construction sites and garages",
"Cordless oscillating multi tool for sanding cutting and scraping tasks",
"Hand saw wood cutting tool with ergonomic handle for carpentry",
"Wall mounted floating wooden shelves for storage and home decoration",
"Bathroom sink mixer tap chrome plated faucet for modern washbasins",
"Smart LED bulb WiFi enabled lighting compatible with home automation",
"Digital laser distance meter tool for measuring room dimensions accurately",
"Portable pressure washer machine for cleaning cars driveways and patios",
"Electric jigsaw tool for precision curved wood cutting projects",
"Garden shovel steel digging tool for landscaping and soil preparation",
"Heavy duty extension power cord with multiple plug outlets",
"LED under cabinet lighting strip for kitchen countertop illumination",
"Cordless leaf blower battery powered tool for garden cleaning",
"Electric angle grinder metal cutting tool with safety guard",
"DIY wall repair kit for fixing cracks holes and drywall damage",
"Paint brush set professional quality brushes for detailed painting",
"Portable ladder aluminum folding ladder for indoor maintenance work",
"Garden rake steel tines tool for leveling soil and collecting leaves",
"Bathroom exhaust fan ventilation system for moisture control",
"Waterproof outdoor extension cable for garden electrical equipment",
"Electric drill bit set for wood metal and masonry drilling",
"Magnetic screwdriver bit holder accessory for power drills",
"Woodworking clamps heavy duty clamps for holding boards during glue up",
"Adjustable stud finder tool for locating wooden studs behind drywall",
"Wall mounted coat rack wooden hanger with metal hooks",
"LED floodlight outdoor security lighting for driveways and gardens",
"Smart thermostat digital temperature controller for home heating",
"Electric soldering iron kit for electronics repair and wiring",
"Pipe cutter tool for cutting copper PVC and plastic pipes",
"Kitchen cabinet handle stainless steel replacement drawer handles",
"Portable tile leveling system clips for perfect tile alignment",
"Cordless drill lithium battery replacement pack compatible with Bosch tools",
"Heavy duty tool storage box organizer for garage tools",
"Portable workbench folding work table for DIY projects",
"Digital multimeter electrical testing device for voltage current resistance",
"Insulated pliers electrician tool for safe electrical work",
"Wall painting stencil kit decorative patterns for home interiors",
"Automatic garden irrigation timer for watering plants efficiently",
"Handheld grout removal tool for tile repair and renovation",
"Steel measuring tape 5 meter construction measurement tool",
"Portable LED headlamp rechargeable lamp for working in dark spaces",
"Electric heat gun tool for paint stripping and shrink wrapping",
"Heavy duty staple gun for upholstery and woodworking projects",
"Adjustable torque wrench precision tool for mechanical tightening",
"Home door lock smart fingerprint enabled security lock",
"Aluminum window sliding track roller replacement kit",
"Carpenter square angle measurement tool for woodworking accuracy",
"Professional caulking gun sealant applicator for bathroom sealing",
"LED mirror light bar for bathroom vanity illumination",
"Portable generator inverter power backup for home emergencies",
"Insulated screwdrivers electrician tool kit for electrical installations",
"Water pump submersible pump for draining water from basements",
"Bathroom towel rack stainless steel wall mounted holder",
"Digital humidity meter indoor air moisture measurement device",
"Electric hot glue gun craft repair adhesive tool",
"Wood chisel carving set for fine woodworking tasks",
"Expandable closet organizer rod adjustable storage solution",
"Bathroom floor drain stainless steel anti odor drain cover",
"Rechargeable portable fan battery powered cooling device",
"Electric wall chaser machine for cutting grooves in walls",
"Magnetic tool holder strip for organizing garage tools",
"PVC pipe fittings set connectors for plumbing installations",
"Electric lawn mower cordless grass cutting machine",
"Garden soil moisture sensor smart irrigation monitor",
"LED garage ceiling light ultra bright workshop lighting",
"Portable cement mixer machine for small construction work",
"DIY epoxy floor coating kit for garage flooring",
"Aluminum spirit level bubble level tool for straight alignment",
"Electric power planer woodworking smoothing tool",
"Heavy duty crowbar pry bar demolition tool",
"Garden wheelbarrow steel tray cart for transporting soil",
"Waterproof junction box electrical outdoor wiring protection",
"Roof repair sealant waterproof adhesive for leak fixing",
"Bathroom vanity cabinet with sink storage unit",
"Electric nail gun pneumatic style tool for carpentry framing",
"Portable dust extractor vacuum for woodworking tools"
]

In [6]:
query = "best cordless drill for home DIY repair"

# Parallel preparation of query-doc pairs
def create_pair(doc):
    return [query, doc]

with ThreadPoolExecutor() as executor:
    pairs = list(executor.map(create_pair, documents))

# GPU warmup
_ = reranker.compute_score(pairs[:8], batch_size=8)

# Run reranking with batching
start = time.time()

scores = reranker.compute_score(
    pairs,
    batch_size=32
)

end = time.time()

Compute Scores: 100%|██████████| 4/4 [00:00<00:00, 25.89it/s]


In [7]:
# Combine results
results = list(zip(documents, scores))
reranked = sorted(results, key=lambda x: x[1], reverse=True)

print("Latency:", round(end-start,3), "seconds")

print("\nTop 10 Results:\n")
for doc, score in reranked[:10]:
    print(round(score,3), "|", doc)

Latency: 0.211 seconds

Top 10 Results:

6.18 | Bosch 20V cordless drill with lithium battery designed for DIY home repair and drilling into wood and drywall
-0.193 | Adjustable torque wrench precision tool for mechanical tightening
-0.298 | Cordless oscillating multi tool for sanding cutting and scraping tasks
-2.252 | Handheld grout removal tool for tile repair and renovation
-2.928 | Hand saw wood cutting tool with ergonomic handle for carpentry
-2.934 | Electric jigsaw tool for precision curved wood cutting projects
-3.006 | Waterproof outdoor extension cable for garden electrical equipment
-3.213 | Electric drill bit set for wood metal and masonry drilling
-3.234 | Cordless leaf blower battery powered tool for garden cleaning
-3.338 | Adjustable stud finder tool for locating wooden studs behind drywall


In [8]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [13]:
pairs = [[query, doc[:256]] for doc in documents]

In [14]:
_ = reranker.compute_score(pairs[:8], batch_size=8)

torch.cuda.synchronize()

Compute Scores: 100%|██████████| 1/1 [00:00<00:00, 96.47it/s]


In [15]:
torch.cuda.synchronize()

start = time.time()

scores = reranker.compute_score(
    pairs,
    batch_size=64
)

torch.cuda.synchronize()

end = time.time()

latency_ms = (end - start) * 1000
print("Reranking latency:", round(latency_ms,2), "ms")

Compute Scores: 100%|██████████| 2/2 [00:00<00:00, 38.79it/s]

Reranking latency: 119.63 ms


In [16]:
results = list(zip(documents, scores))

In [17]:
reranked = sorted(results, key=lambda x: x[1], reverse=True)

In [18]:
for doc, score in reranked[:10]:
    print(round(score,3), "|", doc)

6.176 | Bosch 20V cordless drill with lithium battery designed for DIY home repair and drilling into wood and drywall
-0.193 | Adjustable torque wrench precision tool for mechanical tightening
-0.302 | Cordless oscillating multi tool for sanding cutting and scraping tasks
-2.25 | Handheld grout removal tool for tile repair and renovation
-2.926 | Hand saw wood cutting tool with ergonomic handle for carpentry
-2.936 | Electric jigsaw tool for precision curved wood cutting projects
-3.006 | Waterproof outdoor extension cable for garden electrical equipment
-3.207 | Electric drill bit set for wood metal and masonry drilling
-3.23 | Cordless leaf blower battery powered tool for garden cleaning
-3.336 | Adjustable stud finder tool for locating wooden studs behind drywall
